# Flight Anomaly Live Learning Monitor

This Colab notebook has been simplified so the full demonstration runs from a single code cell. That avoids cross-cell package and import state issues in Google Colab.

In [ ]:
from __future__ import annotations

import math
import random
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
    from sklearn.linear_model import SGDClassifier
    from sklearn.preprocessing import StandardScaler
    SKLEARN_AVAILABLE = True
    SKLEARN_ENSEMBLES_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False
    SKLEARN_ENSEMBLES_AVAILABLE = False

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

print("Environment ready")
print("NumPy:", np.__version__)
try:
    import sklearn
    print("scikit-learn:", sklearn.__version__)
except Exception:
    print("scikit-learn not available; fallback scoring will be used")

project_root = Path("/content/flight_monitor_project")
project_root.mkdir(parents=True, exist_ok=True)

route_csv = """lat,lon
28.5562,77.1000
27.9000,75.5000
26.9000,73.8000
24.7000,72.9000
22.9000,72.3000
20.9000,72.7000
19.0896,72.8656
"""

engine_csv = """engine_temp_c,vibration_g,oil_pressure_psi,hydraulic_psi,fuel_flow_kg_h,avionics_bus_v
690,1.8,71,3050,2350,28.0
"""

airspace_csv = """headline,severity
No active airspace disruptions in planned corridor.,0.05
Routine ATC advisories issued for high traffic sectors.,0.22
Moderate weather cells observed near alternate corridor.,0.40
"""

(project_root / "flight_route_seed.csv").write_text(route_csv, encoding="utf-8")
(project_root / "engine_sensor_baseline.csv").write_text(engine_csv, encoding="utf-8")
(project_root / "airspace_news_seed.csv").write_text(airspace_csv, encoding="utf-8")


def load_route() -> list[dict]:
    rows = []
    for line in route_csv.strip().splitlines()[1:]:
        lat, lon = line.split(",")
        rows.append({"lat": float(lat), "lon": float(lon)})
    return rows


def load_engine_baseline() -> dict:
    values = engine_csv.strip().splitlines()[1].split(",")
    return {
        "engine_temp_c": float(values[0]),
        "vibration_g": float(values[1]),
        "oil_pressure_psi": float(values[2]),
        "hydraulic_psi": float(values[3]),
        "fuel_flow_kg_h": float(values[4]),
        "avionics_bus_v": float(values[5]),
    }


def load_airspace_news() -> list[dict]:
    rows = []
    for line in airspace_csv.strip().splitlines()[1:]:
        headline, severity = line.rsplit(",", 1)
        rows.append({"headline": headline, "severity": float(severity)})
    return rows


@dataclass
class ModelScore:
    score: float
    is_anomaly: bool
    details: dict


class OnlineBinaryAnomalyModel:
    def __init__(self, n_features: int, threshold: float = 0.65):
        self.n_features = n_features
        self.threshold = threshold
        self._fallback_center = np.zeros(n_features)
        self._fallback_var = np.ones(n_features)
        self._fallback_n = 0
        self.scaler = StandardScaler() if SKLEARN_AVAILABLE else None
        self.clf = SGDClassifier(loss="log_loss", alpha=0.0005, random_state=42) if SKLEARN_AVAILABLE else None
        self._initialized = False

    def update(self, x: np.ndarray, y: int) -> float:
        x = x.reshape(1, -1)
        if not SKLEARN_AVAILABLE:
            self._fallback_update(x[0])
            return float(self._fallback_score(x[0]))

        self.scaler.partial_fit(x)
        x_scaled = self.scaler.transform(x)
        classes = np.array([0, 1])
        if not self._initialized:
            self.clf.partial_fit(x_scaled, np.array([y]), classes=classes)
            self._initialized = True
        else:
            self.clf.partial_fit(x_scaled, np.array([y]))
        return float(self.clf.predict_proba(x_scaled)[0][1])

    def score(self, x: np.ndarray) -> float:
        if not SKLEARN_AVAILABLE or not self._initialized:
            return float(self._fallback_score(x))
        x_scaled = self.scaler.transform(x.reshape(1, -1))
        return float(self.clf.predict_proba(x_scaled)[0][1])

    def evaluate(self, x: np.ndarray) -> ModelScore:
        score = self.score(x)
        return ModelScore(score=score, is_anomaly=score >= self.threshold, details={})

    def _fallback_update(self, x: np.ndarray) -> None:
        self._fallback_n += 1
        if self._fallback_n == 1:
            self._fallback_center = x
            return
        delta = x - self._fallback_center
        self._fallback_center += delta / self._fallback_n
        delta2 = x - self._fallback_center
        self._fallback_var += delta * delta2

    def _fallback_score(self, x: np.ndarray) -> float:
        std = np.sqrt(np.maximum(self._fallback_var / max(self._fallback_n - 1, 1), 1e-3))
        z = np.abs((x - self._fallback_center) / std)
        raw = float(np.mean(z))
        return 1.0 - math.exp(-raw / 3.0)


class TinyAutoencoder:
    def __init__(self, input_dim: int, hidden_dim: int = 4, lr: float = 0.001):
        rng = np.random.default_rng(42)
        self.w1 = rng.normal(0, 0.1, size=(input_dim, hidden_dim))
        self.b1 = np.zeros(hidden_dim)
        self.w2 = rng.normal(0, 0.1, size=(hidden_dim, input_dim))
        self.b2 = np.zeros(input_dim)
        self.lr = lr
        self.err_ema = 1.0

    def train_and_score(self, x: np.ndarray) -> float:
        h = np.tanh(x @ self.w1 + self.b1)
        y = h @ self.w2 + self.b2
        err_vec = y - x
        mse = float(np.mean(err_vec ** 2))

        dy = 2.0 * err_vec / len(x)
        dw2 = np.outer(h, dy)
        db2 = dy
        dh = dy @ self.w2.T
        dz1 = dh * (1.0 - h ** 2)
        dw1 = np.outer(x, dz1)
        db1 = dz1

        self.w2 -= self.lr * dw2
        self.b2 -= self.lr * db2
        self.w1 -= self.lr * dw1
        self.b1 -= self.lr * db1

        self.err_ema = 0.95 * self.err_ema + 0.05 * mse
        norm = mse / max(self.err_ema, 1e-6)
        return float(1.0 - math.exp(-norm))


@dataclass
class Waypoint:
    lat: float
    lon: float


class FlightSimulationEngine:
    def __init__(self) -> None:
        self._rng = random.Random(7)
        self.controls = {
            "route_deviation": 0.1,
            "engine_stress": 0.2,
            "weather": 0.1,
            "airspace_risk": 0.2,
        }
        self._injections = {
            "engine_temp_delta": 0.0,
            "vibration_delta": 0.0,
            "force_airspace_shutdown": False,
        }
        self.route = self._build_route()
        self.route_idx = 0
        self.position = Waypoint(self.route[0].lat, self.route[0].lon)
        self.path_model = OnlineBinaryAnomalyModel(n_features=5, threshold=0.65)
        self.engine_model = OnlineBinaryAnomalyModel(n_features=8, threshold=0.7)
        self.airspace_model = OnlineBinaryAnomalyModel(n_features=5, threshold=0.6)
        self.engine_autoencoder = TinyAutoencoder(input_dim=6, hidden_dim=4, lr=0.001)
        self.engine_status = {}
        self.airspace_state = {}
        self.path_state = {}
        self.history = []
        self.compare_history = {}
        self.compare_records = []
        self.static_models = {}
        self.xgb_backend = "xgboost" if XGBOOST_AVAILABLE else "gradient_boosting_fallback"
        self.step_count = 0
        self.reset()

    def reset(self) -> None:
        self.route = self._build_route()
        self.route_idx = 0
        self.position = Waypoint(self.route[0].lat, self.route[0].lon)
        self.history = [{"lat": self.position.lat, "lon": self.position.lon, "anomaly": False}]
        self.compare_history = {
            "live_learning": [{"lat": self.position.lat, "lon": self.position.lon, "anomaly": False}],
            "random_forest": [{"lat": self.position.lat, "lon": self.position.lon, "anomaly": False}],
            "xgboost": [{"lat": self.position.lat, "lon": self.position.lon, "anomaly": False}],
        }
        self.compare_records = []
        self.static_models = self._build_static_baseline_models()
        self.step_count = 0
        self.engine_status = load_engine_baseline()
        self.airspace_state = {
            "open_corridors": 4,
            "restricted_corridors": 0,
            "shutdown_corridors": 0,
            "latest_news": [
                {"headline": n["headline"], "severity": n["severity"], "time": self._iso_now()}
                for n in load_airspace_news()
            ],
        }
        self.path_state = {
            "distance_from_plan_km": 0.0,
            "heading_error_deg": 0.0,
            "speed_kts": 465.0,
        }

    def apply_controls(self, controls: dict, injections: dict) -> None:
        for k in self.controls:
            if k in controls:
                self.controls[k] = float(np.clip(controls[k], 0.0, 1.0))
        for k in self._injections:
            if k in injections:
                if isinstance(self._injections[k], bool):
                    self._injections[k] = bool(injections[k])
                else:
                    self._injections[k] = float(injections[k])

    def step(self) -> None:
        self.step_count += 1
        target_idx = min(self.route_idx + 1, len(self.route) - 1)
        target = self.route[target_idx]

        deviation = self.controls["route_deviation"]
        weather = self.controls["weather"]
        stress = self.controls["engine_stress"]
        air_risk = self.controls["airspace_risk"]

        lat_step = (target.lat - self.position.lat) * 0.25
        lon_step = (target.lon - self.position.lon) * 0.25
        noise_scale = 0.02 + 0.2 * deviation + 0.1 * weather

        self.position.lat += lat_step + self._rng.uniform(-noise_scale, noise_scale)
        self.position.lon += lon_step + self._rng.uniform(-noise_scale, noise_scale)

        if abs(self.position.lat - target.lat) + abs(self.position.lon - target.lon) < 0.15:
            self.route_idx = target_idx

        ref = self.route[self.route_idx]
        distance_km = self._geo_distance_km(self.position, ref)
        heading_error = 180.0 * min(1.0, distance_km / 40.0)
        speed = 470.0 + self._rng.uniform(-8, 8) - 35.0 * weather - 20.0 * deviation
        self.path_state = {
            "distance_from_plan_km": round(distance_km, 3),
            "heading_error_deg": round(heading_error, 2),
            "speed_kts": round(speed, 2),
        }

        self.engine_status["engine_temp_c"] = 685 + 130 * stress + 40 * weather + self._rng.uniform(-8, 8)
        self.engine_status["engine_temp_c"] += self._injections["engine_temp_delta"]
        self.engine_status["vibration_g"] = 1.7 + 2.6 * stress + 0.8 * deviation + self._rng.uniform(-0.15, 0.15)
        self.engine_status["vibration_g"] += self._injections["vibration_delta"]
        self.engine_status["oil_pressure_psi"] = 73 - 22 * stress - 12 * weather + self._rng.uniform(-2.5, 2.5)
        self.engine_status["hydraulic_psi"] = 3050 - 210 * stress - 110 * weather + self._rng.uniform(-30, 30)
        self.engine_status["fuel_flow_kg_h"] = 2300 + 320 * stress + 170 * weather + self._rng.uniform(-35, 35)
        self.engine_status["avionics_bus_v"] = 28.2 - 1.0 * stress + self._rng.uniform(-0.3, 0.3)

        self._update_airspace(air_risk, weather)

        path_x = np.array([distance_km, heading_error, speed, weather, deviation], dtype=float)
        path_label = int(distance_km > 7 or heading_error > 20)
        path_score = self.path_model.update(path_x, path_label)
        rf_score = self._static_model_score("random_forest", path_x)
        xgb_score = self._static_model_score("xgboost", path_x)

        engine_x = np.array([
            self.engine_status["engine_temp_c"],
            self.engine_status["vibration_g"],
            self.engine_status["oil_pressure_psi"],
            self.engine_status["hydraulic_psi"],
            self.engine_status["fuel_flow_kg_h"],
            self.engine_status["avionics_bus_v"],
            stress,
            weather,
        ], dtype=float)
        engine_label = int(
            self.engine_status["engine_temp_c"] > 780
            or self.engine_status["vibration_g"] > 3.9
            or self.engine_status["oil_pressure_psi"] < 42
            or self.engine_status["hydraulic_psi"] < 2700
        )
        engine_score_ml = self.engine_model.update(engine_x, engine_label)
        engine_ae_score = self.engine_autoencoder.train_and_score(engine_x[:6])
        engine_score = 0.65 * engine_score_ml + 0.35 * engine_ae_score

        air_features = self._airspace_features(air_risk)
        air_label = int(self.airspace_state["shutdown_corridors"] > 0 or air_features[2] > 0.5)
        air_score = self.airspace_model.update(np.array(air_features, dtype=float), air_label)

        path_anomaly = path_score >= self.path_model.threshold
        rf_anomaly = rf_score >= self.path_model.threshold
        xgb_anomaly = xgb_score >= self.path_model.threshold
        engine_anomaly = engine_score >= self.engine_model.threshold
        air_anomaly = air_score >= self.airspace_model.threshold

        point = {"lat": round(self.position.lat, 5), "lon": round(self.position.lon, 5)}
        self.history.append({**point, "anomaly": bool(path_anomaly)})
        self.history = self.history[-300:]
        self.compare_history["live_learning"].append({**point, "anomaly": bool(path_anomaly)})
        self.compare_history["random_forest"].append({**point, "anomaly": bool(rf_anomaly)})
        self.compare_history["xgboost"].append({**point, "anomaly": bool(xgb_anomaly)})
        self.compare_history["live_learning"] = self.compare_history["live_learning"][-300:]
        self.compare_history["random_forest"] = self.compare_history["random_forest"][-300:]
        self.compare_history["xgboost"] = self.compare_history["xgboost"][-300:]
        self.compare_records.append({
            "t": self.step_count,
            "label": int(path_label),
            "live_score": float(path_score),
            "rf_score": float(rf_score),
            "xgb_score": float(xgb_score),
            "live_pred": int(path_anomaly),
            "rf_pred": int(rf_anomaly),
            "xgb_pred": int(xgb_anomaly),
        })
        self.compare_records = self.compare_records[-300:]

        self.path_state["anomaly_score"] = round(float(path_score), 4)
        self.path_state["is_anomaly"] = bool(path_anomaly)
        self.engine_status["anomaly_score"] = round(float(engine_score), 4)
        self.engine_status["ml_score"] = round(float(engine_score_ml), 4)
        self.engine_status["ae_score"] = round(float(engine_ae_score), 4)
        self.engine_status["is_anomaly"] = bool(engine_anomaly)
        self.airspace_state["anomaly_score"] = round(float(air_score), 4)
        self.airspace_state["is_anomaly"] = bool(air_anomaly)

        self._injections["engine_temp_delta"] *= 0.85
        self._injections["vibration_delta"] *= 0.85
        self._injections["force_airspace_shutdown"] = False

    def get_state(self) -> dict:
        return {
            "timestamp": self._iso_now(),
            "controls": dict(self.controls),
            "flight": {
                "progress": round(self.route_idx / max(len(self.route) - 1, 1), 3),
                "current_position": {"lat": round(self.position.lat, 5), "lon": round(self.position.lon, 5)},
                "planned_route": [{"lat": p.lat, "lon": p.lon} for p in self.route],
                "actual_path": list(self.history),
                "path_metrics": dict(self.path_state),
            },
            "engine": dict(self.engine_status),
            "airspace": dict(self.airspace_state),
        }

    def get_model_comparison(self) -> dict:
        live_metrics = self._classification_metrics([x["live_pred"] for x in self.compare_records], [x["label"] for x in self.compare_records])
        rf_metrics = self._classification_metrics([x["rf_pred"] for x in self.compare_records], [x["label"] for x in self.compare_records])
        xgb_metrics = self._classification_metrics([x["xgb_pred"] for x in self.compare_records], [x["label"] for x in self.compare_records])
        return {
            "models": {
                "live_learning": {"summary": live_metrics, "series": self._model_series("live_score", "live_pred")},
                "random_forest": {"summary": rf_metrics, "series": self._model_series("rf_score", "rf_pred")},
                "xgboost": {"summary": xgb_metrics, "series": self._model_series("xgb_score", "xgb_pred"), "backend": self.xgb_backend},
            },
            "advantage_gap": {
                "accuracy_live_vs_rf": round(live_metrics["accuracy"] - rf_metrics["accuracy"], 4),
                "accuracy_live_vs_xgb": round(live_metrics["accuracy"] - xgb_metrics["accuracy"], 4),
                "f1_live_vs_rf": round(live_metrics["f1"] - rf_metrics["f1"], 4),
                "f1_live_vs_xgb": round(live_metrics["f1"] - xgb_metrics["f1"], 4),
            },
        }

    def _model_series(self, score_key: str, pred_key: str) -> list[dict]:
        labels = [x["label"] for x in self.compare_records]
        preds = [x[pred_key] for x in self.compare_records]
        running_correct = 0
        series = []
        for idx, row in enumerate(self.compare_records):
            if row[pred_key] == row["label"]:
                running_correct += 1
            precision, recall, f1 = self._prf(preds[: idx + 1], labels[: idx + 1])
            series.append({
                "t": row["t"],
                "score": round(float(row[score_key]), 4),
                "accuracy": round(running_correct / (idx + 1), 4),
                "precision": precision,
                "recall": recall,
                "f1": f1,
            })
        return series

    def _build_static_baseline_models(self) -> dict:
        if not SKLEARN_ENSEMBLES_AVAILABLE:
            return {"random_forest": None, "xgboost": None}
        x_train, y_train = self._generate_training_samples(1200)
        rf = RandomForestClassifier(n_estimators=120, max_depth=7, min_samples_leaf=2, random_state=42)
        rf.fit(x_train, y_train)
        if XGBOOST_AVAILABLE:
            xgb = XGBClassifier(
                n_estimators=120,
                max_depth=4,
                learning_rate=0.08,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                random_state=42,
                eval_metric="logloss",
            )
        else:
            xgb = GradientBoostingClassifier(n_estimators=180, learning_rate=0.06, max_depth=3, random_state=42)
        xgb.fit(x_train, y_train)
        return {"random_forest": rf, "xgboost": xgb}

    def _generate_training_samples(self, n_samples: int) -> tuple[np.ndarray, np.ndarray]:
        rows, labels = [], []
        for _ in range(n_samples):
            deviation = self._rng.uniform(0.0, 1.0)
            weather = self._rng.uniform(0.0, 1.0)
            distance = max(0.0, self._rng.gauss(2.0 + 9.0 * deviation + 7.0 * weather, 2.8))
            heading = min(180.0, 1.8 * distance + self._rng.uniform(-6.0, 6.0))
            speed = 470.0 + self._rng.uniform(-12.0, 12.0) - 35.0 * weather - 22.0 * deviation
            label = int(distance > 7.0 or heading > 20.0)
            rows.append([distance, heading, speed, weather, deviation])
            labels.append(label)
        return np.array(rows, dtype=float), np.array(labels, dtype=int)

    def _static_model_score(self, model_key: str, x: np.ndarray) -> float:
        model = self.static_models.get(model_key)
        if model is None:
            return self._heuristic_static_score(x)
        try:
            return float(model.predict_proba(x.reshape(1, -1))[0][1])
        except Exception:
            pred = int(model.predict(x.reshape(1, -1))[0])
            return float(0.78 if pred else 0.22)

    @staticmethod
    def _heuristic_static_score(x: np.ndarray) -> float:
        distance, heading, speed, weather, deviation = x.tolist()
        raw = (
            0.44 * min(distance / 10.0, 1.6)
            + 0.38 * min(heading / 30.0, 1.5)
            + 0.10 * max(0.0, (460.0 - speed) / 55.0)
            + 0.08 * ((weather + deviation) / 2.0)
        )
        return float(max(0.0, min(1.0, raw)))

    @staticmethod
    def _classification_metrics(preds: list[int], labels: list[int]) -> dict:
        if not labels:
            return {"accuracy": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0}
        correct = sum(int(p == y) for p, y in zip(preds, labels))
        precision, recall, f1 = FlightSimulationEngine._prf(preds, labels)
        return {"accuracy": round(correct / len(labels), 4), "precision": precision, "recall": recall, "f1": f1}

    @staticmethod
    def _prf(preds: list[int], labels: list[int]) -> tuple[float, float, float]:
        tp = sum(int(p == 1 and y == 1) for p, y in zip(preds, labels))
        fp = sum(int(p == 1 and y == 0) for p, y in zip(preds, labels))
        fn = sum(int(p == 0 and y == 1) for p, y in zip(preds, labels))
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        f1 = 2 * precision * recall / max(precision + recall, 1e-12)
        return round(precision, 4), round(recall, 4), round(f1, 4)

    def _update_airspace(self, risk: float, weather: float) -> None:
        open_corr = max(1, 4 - int(risk * 2.5))
        restricted = int(risk * 3 + weather * 2 + self._rng.random())
        shutdown = 1 if (self._rng.random() < (0.03 + 0.25 * risk + 0.1 * weather)) else 0
        if self._injections["force_airspace_shutdown"]:
            shutdown = max(shutdown, 1)
        self.airspace_state["open_corridors"] = max(0, open_corr)
        self.airspace_state["restricted_corridors"] = max(0, restricted)
        self.airspace_state["shutdown_corridors"] = shutdown
        if shutdown:
            headline, sev = "Urgent NOTAM: Temporary corridor shutdown due to geopolitical or weather event.", 0.95
        elif restricted > 1:
            headline, sev = "ATC advisory: Multiple corridors under temporary restrictions.", 0.65
        else:
            headline, sev = "No severe restriction bulletin in active corridor.", 0.1 + 0.2 * risk
        news = {"headline": headline, "severity": round(float(sev), 3), "time": self._iso_now()}
        items = self.airspace_state.get("latest_news", [])
        items.insert(0, news)
        self.airspace_state["latest_news"] = items[:10]

    def _airspace_features(self, risk: float) -> list[float]:
        items = self.airspace_state["latest_news"]
        avg_severity = float(np.mean([x["severity"] for x in items])) if items else 0.0
        return [
            float(self.airspace_state["open_corridors"]),
            float(self.airspace_state["restricted_corridors"]),
            float(self.airspace_state["shutdown_corridors"]),
            avg_severity,
            float(risk),
        ]

    def _build_route(self) -> list[Waypoint]:
        return [Waypoint(x["lat"], x["lon"]) for x in load_route()]

    @staticmethod
    def _geo_distance_km(a: Waypoint, b: Waypoint) -> float:
        dx = (a.lat - b.lat) * 111.0
        dy = (a.lon - b.lon) * 85.0
        return math.sqrt(dx * dx + dy * dy)

    @staticmethod
    def _iso_now() -> str:
        return datetime.now(timezone.utc).isoformat()


engine = FlightSimulationEngine()
engine.apply_controls(
    {
        "route_deviation": 0.35,
        "engine_stress": 0.55,
        "weather": 0.30,
        "airspace_risk": 0.45,
    },
    {
        "engine_temp_delta": 25,
        "vibration_delta": 0.6,
        "force_airspace_shutdown": False,
    },
)

for step in range(90):
    if step == 45:
        engine.apply_controls({}, {"force_airspace_shutdown": True})
    engine.step()

state = engine.get_state()
compare = engine.get_model_comparison()
planned = state["flight"]["planned_route"]
actual = state["flight"]["actual_path"]
anoms = [p for p in actual if p["anomaly"]]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].plot([p["lon"] for p in planned], [p["lat"] for p in planned], "--", linewidth=2, label="Planned Route")
axes[0].plot([p["lon"] for p in actual], [p["lat"] for p in actual], linewidth=2.5, label="Actual Path")
if anoms:
    axes[0].scatter([p["lon"] for p in anoms], [p["lat"] for p in anoms], c="red", s=28, label="Detected Path Anomaly")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_title("Flight Route Monitoring Output")
axes[0].legend()
axes[0].grid(alpha=0.3)

live_series = compare["models"]["live_learning"]["series"]
rf_series = compare["models"]["random_forest"]["series"]
xgb_series = compare["models"]["xgboost"]["series"]
axes[1].plot([x["t"] for x in live_series], [x["accuracy"] for x in live_series], label="Live Learning Accuracy", linewidth=2)
axes[1].plot([x["t"] for x in rf_series], [x["accuracy"] for x in rf_series], label="Random Forest Accuracy", linewidth=2)
axes[1].plot([x["t"] for x in xgb_series], [x["accuracy"] for x in xgb_series], label="XGBoost/Fallback Accuracy", linewidth=2)
axes[1].set_xlabel("Simulation Step")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Model Comparison Across the Simulation")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Project files mirrored to:", project_root)
print("Current flight progress:", state["flight"]["progress"])
print("Path metrics:", state["flight"]["path_metrics"])
print("Engine summary:", {k: state["engine"][k] for k in ["engine_temp_c", "vibration_g", "anomaly_score", "is_anomaly"]})
print("Airspace summary:", {k: state["airspace"][k] for k in ["open_corridors", "restricted_corridors", "shutdown_corridors", "anomaly_score", "is_anomaly"]})
